In [1]:
import argparse
import math
import sqlite3
import yaml
import os
import sys
import json
from multiprocessing import Pool

import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
from mol_utils import MolPreprocessor

## SQLite Conditions
- `standard_units`: "IC50", "Inhibition", "activity", "INH", "Ki", "Kd", "EC50"
- `relation`: "=", ">", "<", "<=", ">="; records with other relations are almost non-existent
- `assay_type`: "B" (restrict to binding assays)
- `target_type`: "SINGLE PROTEIN" (restrict to single proteins)
- `standard_value`: exclude missing values

In [4]:
con = sqlite3.connect(f"/home/takutoyo/Database/ChEMBL/chembl_34/chembl_34_sqlite/chembl_34.db")

sql = '''
SELECT
    activity_id,
    act.assay_id                                 AS assay_id,
    t.chembl_id                                  AS target_chembl_id,
    t.pref_name                                  AS target_name,
    t.organism                                   AS target_organism,
    cs.accession                                 AS target_accession,
    cs.sequence                                  AS target_sequence,
    md.chembl_id                                 AS compound_chembl_id,
    cs.canonical_smiles                          AS smiles,
    standard_type,
    relation,
    standard_value,
    standard_units,
    pchembl_value,
    data_validity_comment,
    protein_classification.protein_class_desc    AS protein_class_desc
FROM
    activities act
    JOIN molecule_dictionary md
        ON act.molregno = md.molregno
    JOIN compound_structures cs
        ON md.molregno = cs.molregno
    JOIN assays a
        ON act.assay_id = a.assay_id
    JOIN target_dictionary t
        ON a.tid = t.tid
    JOIN target_components tc
        ON t.tid = tc.tid
    JOIN component_sequences cs
        ON tc.component_id = cs.component_id
    JOIN component_class cc
        ON tc.component_id = cc.component_id
    JOIN protein_classification
        ON cc.protein_class_id = protein_classification.protein_class_id
WHERE
    standard_type IN ('IC50', 'Inhibition', 'activity', 'INH', 'Ki', 'Kd', 'EC50')
    AND relation     IN ('=', '>', '<', '>=', '<=')
    AND (
        standard_units = 'uM'
        OR standard_units = 'nM'
        OR standard_units = 'mM'
    )
    AND a.assay_type     = 'B'
    AND t.target_type    = 'SINGLE PROTEIN'
    AND act.standard_value IS NOT NULL;
'''

df = pd.read_sql(sql, con)
con.close()

`data_validity_comment`: include only records where it is null (no issues) or `"Manually validated"`.


In [6]:
print(df.shape)
df.head()

(1748462, 16)


,activity_id,assay_id,target_chembl_id,target_name,target_organism,target_accession,target_sequence,compound_chembl_id,smiles,standard_type,relation,standard_value,standard_units,pchembl_value,data_validity_comment,protein_class_desc
0,32534,2560,CHEMBL322,Serotonin 2a (5-HT2a) receptor,Rattus norvegicus,P14842,MEILCEDNISLSSIPNSLMQLGDGPRLYHNDFNSRDANTSEASNWT...,CHEMBL6710,C[C@@H](N)Cc1c2c(c(Br)c3c1OCC3)OCC2.Cl,EC50,=,8.38,nM,8.08,None,membrane receptor 7tm1 smallmol monoamine r...
1,33542,151700,CHEMBL5136,Platelet activating factor receptor,Cavia porcellus,P21556,MELNSSSRVDSEFRYTLFPIVYSIIFVLGIIANGYVLWVFARLYPS...,CHEMBL269131,CCCN(CCC)C(=O)OCC1CN(C(=O)c2cc(OC)c(OC)c(OC)c2...,EC50,=,680.00,nM,6.17,None,membrane receptor 7tm1 smallmol lipid-like ...
2,33546,151700,CHEMBL5136,Platelet activating factor receptor,Cavia porcellus,P21556,MELNSSSRVDSEFRYTLFPIVYSIIFVLGIIANGYVLWVFARLYPS...,CHEMBL6203,COc1cc(C(=O)N2CCN(C(=O)c3cc(OC)c(OC)c(OC)c3)C(...,EC50,=,610.00,nM,6.21,None,membrane receptor 7tm1 smallmol lipid-like ...
3,38323,1598,CHEMBL1898,Serotonin 1b (5-HT1b) receptor,Homo sapiens,P28222,MEEPGAQCAPPPPAGSETWVPQANLSSAPSQNCSAKDYIYQDSISL...,CHEMBL323986,COc1ccc(NC(=O)N2CCN(c3ccc(C)cc3C)CC2)cc1N1CCN(...,EC50,=,1.90,nM,8.72,None,membrane receptor 7tm1 smallmol monoamine r...
4,39585,151700,CHEMBL5136,Platelet activating factor receptor,Cavia porcellus,P21556,MELNSSSRVDSEFRYTLFPIVYSIIFVLGIIANGYVLWVFARLYPS...,CHEMBL6217,COc1cc(C(=O)N2CCN(C(=O)c3cc(OC)c(OC)c(OC)c3)C(...,EC50,=,400.00,nM,6.40,None,membrane receptor 7tm1 smallmol lipid-like ...


In [7]:
df1 = df[df["data_validity_comment"].isnull()] # null = no problem
df2 = df[df["data_validity_comment"]=="Manually validated"]
df = pd.concat([df1, df2], axis=0).sort_index().drop("data_validity_comment", axis=1).reset_index(drop=True)

In [8]:
len(df)

1686350

## Binary Classification Label Assignment

In [ ]:
def binalize(row, p_threshold):
    """
    Binarizes a given row of data based on the specified threshold.

    Args:
        row (pd.Series): A single row from a DataFrame, containing the data to be binarized.
        p_threshold (int): The threshold value for binarization.

    Returns:
        int or float: The binarized value (1, 0, or math.nan).

    Raises:
        ValueError: If an unsupported unit or relation is encountered.
    """
    if row["relation"] == "=":
        if row["standard_units"] == "uM":
            if row["standard_value"] <= 10**(-p_threshold+6):
                t = 1
            else:
                t = 0 
        elif row["standard_units"] == "nM":
            if row["standard_value"] <= 10**(-p_threshold+9):
                t = 1 
            else:
                t = 0 
        elif row["standard_units"] == "mM":
            if row["standard_value"] <= 10**(-p_threshold+3):
                t = 1 
            else:
                t = 0
        else:
            raise ValueError("%s is not allowed" % (row["standard_units"]))
    
    elif row["relation"] == ">" or row["relation"] == ">=":
        if row["standard_units"] == "uM":
            if row["standard_value"] <= 10**(-p_threshold+6):
                t = math.nan
            else:
                t = 0 
        elif row["standard_units"] == "nM":
            if row["standard_value"] <= 10**(-p_threshold+9):
                t = math.nan
            else:
                t = 0 
        elif row["standard_units"] == "mM":
            if row["standard_value"] <= 10**(-p_threshold+3):
                t = math.nan
            else:
                t = 0
        else:
            raise ValueError("%s is not allowed" % (row["standard_units"]))
    
    elif row["relation"] == "<" or row["relation"] == "<=":
        if row["standard_units"] == "uM":
            if row["standard_value"] <= 10**(-p_threshold+6):
                t = 1
            else:
                t = math.nan 
        elif row["standard_units"] == "nM":
            if row["standard_value"] <= 10**(-p_threshold+9):
                t = 1
            else:
                t = math.nan 
        elif row["standard_units"] == "mM":
            if row["standard_value"] <= 10**(-p_threshold+3):
                t = 1
            else:
                t = math.nan
        else:
            raise ValueError("%s is not allowed" % (row["standard_units"]))
    else:
        raise ValueError("%s is not allowed" % (row["relation"]))
        
    return t


def Binalize(df):
    """
    Applies binarization to each row of the DataFrame based on multiple thresholds.

    Args:
        df (pd.DataFrame): The DataFrame containing the data to be binarized.

    Returns:
        pd.DataFrame: The DataFrame with additional columns for binarized values.
    """
    t_10us =[]
    t_1us = []
    t_100ns = []
    
    for i in tqdm(range(len(df))):
        row = df.iloc[i]
        t_10us.append(binalize(row, 5))
        t_1us.append(binalize(row, 6))
        t_100ns.append(binalize(row, 7))

    df["t_10u"] = t_10us
    df["t_1u"] = t_1us
    df["t_100n"] = t_100ns
    return df

In [12]:
df = Binalize(df)
df = df.dropna(subset=["t_10u", "t_1u", "t_100n"], how="all")

  0%|          | 0/1686350 [00:00<?, ?it/s]

100%|██████████| 1686350/1686350 [01:54<00:00, 14682.86it/s]


In [ ]:
len(df)

化合物の前処理。処理内容は
- smilesの正規化
- フラグメントの選択 (すなわち塩の除去)
- 重原子数``minimum_heavy_atoms`` 未満の化合物を除去。極端に小さい化合物や、フラグメント選択後に金属や塩、溶媒などが残っている化合物を除去するため

In [7]:
mol_preprocessor = MolPreprocessor(df["smiles"].tolist())
df["smiles"] = mol_preprocessor.preprocess(num=32, minimum_heavy_atoms=4)
df = df.dropna(subset=["smiles", "target_sequence"])

100%|██████████| 1676139/1676139 [02:19<00:00, 12043.22it/s]


### タンパク質情報の付与
``../input/uniprot-reviewed-true-2023.02.23.tsv``を用いて、タンパク質のファミリー情報を付与する

In [8]:
from protein_utils import preprocess_sequence, classify_family

In [9]:
#タンパク質配列の前処理
df_target = df[["target_chembl_id", "target_name", "target_organism", "target_accession", "target_sequence", "protein_class_desc"]].drop_duplicates().reset_index(drop=True)
vocab = ["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", 
         "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "X"]
df_target = preprocess_sequence(df_target, vocab)

100%|██████████| 6319/6319 [00:01<00:00, 4524.91it/s]


In [10]:
uniprot_df = pd.read_csv(f'../input/uniprot-reviewed-true-2023.02.23.tsv', sep='\t')

In [11]:
# Uniprot ID, Protein families, 3D構造の情報を付与
df_target = pd.merge(df_target, uniprot_df[['Entry', 'Protein families', "3D"]], left_on='target_accession', right_on='Entry', how='left')
fam_dicts = [classify_family(family) for family in df_target['Protein families']]
fam_df = pd.concat(fam_dicts, axis=0).reset_index(drop=True)
df_target = pd.concat([df_target, fam_df], axis=1)
df = pd.merge(df, df_target.drop("Entry", axis=1))

In [12]:
df

,activity_id,assay_id,target_chembl_id,target_name,target_organism,target_accession,target_sequence,compound_chembl_id,smiles,standard_type,...,pchembl_value,protein_class_desc,t_10u,t_1u,t_100n,Protein families,3D,superfamily,family,subfamily
0,32534,2560,CHEMBL322,Serotonin 2a (5-HT2a) receptor,Rattus norvegicus,P14842,MEILCEDNISLSSIPNSLMQLGDGPRLYHNDFNSRDANTSEASNWT...,CHEMBL6710,C[C@@H](N)Cc1c2c(c(Br)c3c1OCC3)OCC2,EC50,...,8.08,membrane receptor 7tm1 smallmol monoamine r...,1.0,1.0,1.0,G-protein coupled receptor 1 family,X-ray crystallography (1),NaN,G-protein coupled receptor 1 family,NaN
1,40942,2560,CHEMBL322,Serotonin 2a (5-HT2a) receptor,Rattus norvegicus,P14842,MEILCEDNISLSSIPNSLMQLGDGPRLYHNDFNSRDANTSEASNWT...,CHEMBL268542,C[C@H](N)Cc1c2c(c(C(F)(F)F)c3c1OCC3)OCC2,EC50,...,7.52,membrane receptor 7tm1 smallmol monoamine r...,1.0,1.0,1.0,G-protein coupled receptor 1 family,X-ray crystallography (1),NaN,G-protein coupled receptor 1 family,NaN
2,42177,2560,CHEMBL322,Serotonin 2a (5-HT2a) receptor,Rattus norvegicus,P14842,MEILCEDNISLSSIPNSLMQLGDGPRLYHNDFNSRDANTSEASNWT...,CHEMBL263078,C[C@H](N)Cc1c2c(c(Br)c3c1OCC3)OCC2,EC50,...,7.46,membrane receptor 7tm1 smallmol monoamine r...,1.0,1.0,1.0,G-protein coupled receptor 1 family,X-ray crystallography (1),NaN,G-protein coupled receptor 1 family,NaN
3,53484,2560,CHEMBL322,Serotonin 2a (5-HT2a) receptor,Rattus norvegicus,P14842,MEILCEDNISLSSIPNSLMQLGDGPRLYHNDFNSRDANTSEASNWT...,CHEMBL6737,COc1ccc(OC)c(C[C@@H](C)N)c1,EC50,...,NaN,membrane receptor 7tm1 smallmol monoamine r...,NaN,0.0,0.0,G-protein coupled receptor 1 family,X-ray crystallography (1),NaN,G-protein coupled receptor 1 family,NaN
4,59600,2560,CHEMBL322,Serotonin 2a (5-HT2a) receptor,Rattus norvegicus,P14842,MEILCEDNISLSSIPNSLMQLGDGPRLYHNDFNSRDANTSEASNWT...,CHEMBL6907,C[C@H](N)Cc1c2ccoc2c(C(F)(F)F)c2ccoc12,EC50,...,7.92,membrane receptor 7tm1 smallmol monoamine r...,1.0,1.0,1.0,G-protein coupled receptor 1 family,X-ray crystallography (1),NaN,G-protein coupled receptor 1 family,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1675606,25033248,2282172,CHEMBL4802065,Monocarboxylate transporter 4,Mus musculus,P57787,MGGAVVDEGPTGIKAPDGGWGWAVLFGCFIITGFSYAFPKAVSVFF...,CHEMBL5282978,CC1(CCCCCC(=O)NCCOCCOCCOCCOc2ccnc3c(S(=O)(=O)N...,Ki,...,6.52,unclassified,1.0,1.0,0.0,"Major facilitator superfamily, Monocarboxylate...",NaN,Major facilitator superfamily,Monocarboxylate porter (TC 2.A.1.13) family,NaN
1675607,25046574,2286030,CHEMBL2834,Protein tyrosine phosphatase receptor type F,Rattus norvegicus,Q64604,MAPEPAPGRRMVPLVPALVMLGLMAGAHGDSKPVFVKVPEDQIGLS...,CHEMBL5269879,O=C(CCc1ccc(C(F)(F)P(=O)(O)O)cc1)Nc1cc(NC(=O)C...,Ki,...,4.00,enzyme phosphatase protein phosphatase tyr ...,0.0,0.0,0.0,"Protein-tyrosine phosphatase family, Receptor ...",NaN,NaN,Protein-tyrosine phosphatase family,Receptor class 2A subfamily
1675608,25090487,2297192,CHEMBL5169225,Proline--tRNA ligase,Escherichia coli,P16659,MRTSQYLLSTLKETPADAEVISHQLMLRAGMIRKLASGLYTWLPTG...,CHEMBL5279127,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COS(=O)(=O)NC(=O)[...,Ki,...,6.31,enzyme ligase,1.0,1.0,0.0,"Class-II aminoacyl-tRNA synthetase family, Pro...",NaN,NaN,Class-II aminoacyl-tRNA synthetase family,ProS type 1 subfamily
1675609,25090488,2297192,CHEMBL5169225,Proline--tRNA ligase,Escherichia coli,P16659,MRTSQYLLSTLKETPADAEVISHQLMLRAGMIRKLASGLYTWLPTG...,CHEMBL1163086,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COS(=O)(=O)NC(=O)[...,Ki,...,8.37,enzyme ligase,1.0,1.0,1.0,"Class-II aminoacyl-tRNA synthetase family, Pro...",NaN,NaN,Class-II aminoacyl-tRNA synthetase family,ProS type 1 subfamily


In [13]:
df.isnull().sum()

activity_id                 0
assay_id                    0
target_chembl_id            0
target_name                 0
target_organism             0
target_accession            0
target_sequence             0
compound_chembl_id          0
smiles                      0
standard_type               0
relation                    0
standard_value              0
standard_units              0
pchembl_value          323362
protein_class_desc          0
t_10u                  138784
t_1u                    38021
t_100n                  18025
Protein families       117214
3D                     368556
superfamily           1222189
family                 118887
subfamily             1010009
dtype: int64

In [14]:
df.to_csv("./chembl_34_activity.csv")